# 03 影片生成（MiniMax H3）

**執行階段 > 全部執行**：掛載 Drive → 啟動 ComfyUI → 依 Prompt 生成首幀（Z-Image）→ H3 生成影片（含音訊）→ MP4 存到 `AI-Workflow/outputs/videos/`，預覽圖在 `outputs/previews/`。

需要 **A100**。第一次執行會下載模型（H3 約 43 GB、Z-Image 約 21 GB）並存進工作區。

H3 沒有負面詞，所以 Prompt 會經過關鍵字過濾（命中就拒絕，沒有略過的選項），首幀只能是本專案生成的圖，或由你聲明來源的 `inputs/` 圖片。不能使用真人照片。

In [ ]:
#@title 1. 掛載 Google Drive，載入 Core { display-mode: "form" }
ROOT = "/content/drive/MyDrive/AI-Workflow"  #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import sys
sys.path.insert(0, ROOT)
from controller import notebook as nb
session = nb.connect(ROOT, mount=False)

In [ ]:
#@title 2. 啟動 ComfyUI { display-mode: "form" }
stats = nb.start_comfyui(session)
nb.check_models(session, "minimax-h3-basic")

In [ ]:
#@title 3. 生成影片 { display-mode: "form" }
#@markdown ### Workflow
workflow = "minimax-h3-basic"  #@param ["minimax-h3-basic"] {allow-input: true}
#@markdown ### Simple
#@markdown Prompt（英文描述畫面與動作）
prompt = "A cup of hot coffee on a wooden table beside a window in soft morning light. Thin steam rises slowly from the cup. The camera remains static in a locked-off shot."  #@param {type:"string"}
#@markdown Duration（秒，4–15）
duration = 5  #@param {type:"slider", min:4, max:15, step:1}
#@markdown 比例（只在首幀由 Z-Image 生成時使用）
aspect = "9:16"  #@param ["9:16", "16:9", "1:1", "3:4", "4:3", "2:3", "3:2"]
#@markdown ### 首幀
#@markdown `generate`：先用 Z-Image 依 Prompt 生一張首幀。`job`：用已完成的圖片 job。`file`：用 `AI-Workflow/inputs/` 裡的圖。
first_frame_source = "generate"  #@param ["generate", "job", "file"]
#@markdown 來源是 `job` 時填 Job ID：
first_frame_job_id = ""  #@param {type:"string"}
#@markdown 來源是 `file` 時填路徑（例如 `inputs/frame.png`），並且**必須**勾選下面的聲明：
first_frame_file = ""  #@param {type:"string"}
#@markdown 這張圖是本專案生成的虛構成人角色，或畫面中沒有人物；**不是真人照片**。
first_frame_attested = False  #@param {type:"boolean"}
first_frame = nb.input_ref(first_frame_source, first_frame_job_id, first_frame_file, first_frame_attested)
#@markdown ---
#@markdown ### Advanced（打勾後，下面的欄位才會生效；0 或空白 = 用預設值）
use_advanced = False  #@param {type:"boolean"}
#@markdown 首幀 Prompt（留空就用上面的 Prompt 生首幀）
first_frame_prompt = ""  #@param {type:"string"}
#@markdown Soundscape
soundscape = "Soft ambient sound that matches the scene."  #@param {type:"string"}
#@markdown Music（N/A = 不要配樂）
music = "N/A"  #@param {type:"string"}
#@markdown Steps
steps = 8  #@param {type:"integer"}
#@markdown Seed（-1 = 隨機）
seed = -1  #@param {type:"integer"}
#@markdown Sampler
sampler = "euler"  #@param ["euler"]
#@markdown Scheduler
scheduler = "beta"  #@param ["beta", "simple"]
#@markdown Turbo LoRA 強度
lora_strength = 1.0  #@param {type:"number"}
#@markdown MP4 CRF
crf = 20  #@param {type:"integer"}
#@markdown 其他參數（JSON，例如 `{"steps": 12}`；可用的名稱見 `nb.describe(session, workflow)`）
extra_json = "{}"  #@param {type:"string"}
simple = dict(prompt=prompt, duration=duration, aspect=aspect)
advanced = dict(first_frame_prompt=first_frame_prompt, soundscape=soundscape, music=music, steps=steps, seed=seed, sampler=sampler, scheduler=scheduler, lora_strength=lora_strength, crf=crf) if use_advanced else None
result = nb.generate(session, workflow, simple, advanced, extra=extra_json if use_advanced else None, inputs={"first_frame": first_frame})

In [ ]:
#@title 4. （選用）繼續等待 Agent 的 job { display-mode: "form" }
#@markdown 勾選後，這本 notebook 會像 01 一樣留著執行佇列裡的 job。
wait_for_jobs = False  #@param {type:"boolean"}
#@markdown 佇列空了這麼多分鐘後停止等待：
idle_minutes = 10  #@param {type:"integer"}
#@markdown 停止後釋放 runtime（不再消耗運算單元）。取消勾選的話 runtime 會繼續計費，要自己中斷。
release_runtime = True  #@param {type:"boolean"}
if wait_for_jobs:
    summary = nb.serve(session, idle_minutes, release_runtime)